# Sesión 2 · AC_PR_2_1 · Aliasing y decisión del intervalo de registro
**Sistemas de adquisición de datos · TSU Electrónica y Pruebas · Jabil · 2027**

No necesitas programar. Ejecuta cada celda con el botón ▶ en orden y cambia solo los campos del formulario (a la derecha de cada celda). Copia los resultados en tu hoja de entrega.

In [ ]:
# @title Preparación (ejecuta esta celda primero; no hay nada que cambiar)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (11, 4), "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})

# Datos de la sesión (vienen de la fuente única del curso, datos2.py)
LECTURAS_POR_CICLO = 10
DIA_S = 86400
DT_LOGGER_S = 60      # intervalo del logger del equipo [s]
ESCENARIOS = [[1, "Almacén de componentes sensibles a humedad", "HR", "ciclo del aire acondicionado", 900, 7], [2, "Refrigerador de pasta de soldadura", "T", "ciclo del compresor", 1200, 7], [3, "Cuarto de servidores", "T", "ciclo del aire acondicionado", 480, 7], [4, "Línea SMT, ambiente", "T", "cambio de turno", 28800, 7], [5, "Oficina con ventana al poniente", "T", "ciclo día–noche", 86400, 7], [6, "Bodega de producto terminado", "HR", "ciclo día–noche", 86400, 7], [7, "Área de empaque junto a la puerta de andén", "T", "apertura de la puerta", 300, 7], [8, "Cámara de prueba de humedad con control on-off", "HR", "ciclo del control", 90, 7], [9, "Gabinete de control con ventilador por termostato", "T", "encendido del ventilador", 100, 7], [10, "Recámara del alumno", "T", "ciclo día–noche", 86400, 7]]
# Almacén: ciclo diario + ciclo del aire acondicionado + ruido pequeño
ALM = dict(media=23.0, ampl_dia=1.5, ampl_ac=0.6,
           periodo_ac_min=15, ruido=0.05)

def temperatura_almacen(t_min):
    rng = np.random.default_rng(7)
    dia = ALM["ampl_dia"] * np.sin(2 * np.pi * (t_min / 1440 - 0.3))
    ac = ALM["ampl_ac"] * np.sin(2 * np.pi * t_min / ALM["periodo_ac_min"])
    return ALM["media"] + dia + ac + ALM["ruido"] * rng.standard_normal(t_min.size)

def frec_aparente(f, fs):
    """Frecuencia que muestra el registro: |f − k·fs|, k = entero más cercano a f/fs."""
    return abs(f - round(f / fs) * fs)

def veredicto(dt, P):
    if dt <= P / LECTURAS_POR_CICLO:
        return "Suficiente"
    if dt < P / 2:
        return "Lo detecta, pero lo describe mal"
    return "Aliasing: el registro miente"

print("Listo. Sigue con el bloque A.")


## Bloque A · Un registro que miente
El aire acondicionado del almacén cicla cada **15 min**. Ejecuta la celda con intervalo de 10 min, escribe el nombre de tu equipo y descarga la gráfica (clic derecho → Guardar imagen). Mide en la gráfica naranja cuántos minutos dura un ciclo.

Después mueve el deslizador a 1, 5, 7, 8 y 10 min. ¿Cuál es el intervalo más largo que no produce aliasing? Anótalo en tu hoja de entrega.

In [ ]:
# @title Bloque A · El almacén registrado con distintos intervalos
nombre_equipo = "Equipo 1"  # @param {type:"string"}
intervalo_min = 10  # @param {type:"slider", min:1, max:20, step:1}

t = np.arange(0, 180, 1 / 6)                 # 3 horas, una lectura "real" cada 10 s
T = temperatura_almacen(t)
ts = np.arange(0, 180 + 1e-9, intervalo_min)
Ts = np.interp(ts, t, T)

fig, ax = plt.subplots()
ax.plot(t, T, color="#9DB7D5", lw=1.2, label="Temperatura real")
ax.plot(ts, Ts, "-o", color="#D9822B", lw=2.2, ms=6, label=f"Registro cada {intervalo_min} min")
ax.set_xlabel("Tiempo [min]"); ax.set_ylabel("Temperatura [°C]"); ax.set_xticks(range(0, 181, 15))
ax.set_title(f"{nombre_equipo} · almacén registrado cada {intervalo_min} min")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
ax.grid(alpha=0.3)
plt.show()

f = 60 / ALM["periodo_ac_min"]; fs = 60 / intervalo_min
fa = frec_aparente(f, fs)
print(f"Ciclo real del aire: {ALM['periodo_ac_min']} min  ({f:g} ciclos/h)")
print(f"Registro: {fs:g} lecturas/h  → límite de Nyquist {fs/2:g} ciclos/h")
print("Frecuencia que muestra el registro:", f"{fa:g} ciclos/h" + (f"  → periodo aparente {60/fa:.0f} min" if fa > 0 else ""))
print("Veredicto:", veredicto(intervalo_min, ALM["periodo_ac_min"]))


## Bloque B · Tu escenario
Elige tu número de equipo. El intervalo del logger es el del equipo que van a usar; no lo cambies salvo que el profesor lo indique.

In [ ]:
# @title Bloque B · Tu escenario
equipo = 1  # @param {type:"slider", min:1, max:10, step:1}
intervalo_logger_s = 60  # @param {type:"number"}

eq, lugar, variable, fenomeno, P, dias = ESCENARIOS[equipo - 1]
print(f"Equipo {eq}: {lugar}")
print(f"Variable: {variable}. Fenómeno más rápido: {fenomeno}, periodo P = {P} s")

# Señal ilustrativa: cuatro ciclos del fenómeno, leídos muy fino y por el logger
t = np.linspace(0, 4 * P, 4000)
y = np.sin(2 * np.pi * t / P)
ts = np.arange(0, 4 * P + 1e-9, intervalo_logger_s)
fig, ax = plt.subplots()
ax.plot(t / 60, y, color="#9DB7D5", lw=1.6, label="Fenómeno real")
ax.plot(ts / 60, np.sin(2 * np.pi * ts / P), "-o", color="#D9822B", ms=5, lw=1.8,
        label=f"Logger cada {intervalo_logger_s:g} s")
ax.set_xlabel("Tiempo [min]"); ax.set_yticks([])
ax.set_title(f"Equipo {eq} · {fenomeno} (P = {P} s)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
plt.show()


## Bloque C · Verificación contra tu hoja de cálculo
Antes de ejecutar, calcula a mano en la pestaña «Práctica» de tu hoja (columna B). Luego pega los valores de Colab en la columna E: la columna F debe marcar ✔ en los cinco renglones.

In [ ]:
# @title Bloque C · Tabla resumen (cópiala en la pestaña «Práctica», columna E)
D_s = dias * DIA_S
filas = [
    ("Δt máximo por Nyquist = P/2", P / 2, "s"),
    (f"Δt recomendado = P/{LECTURAS_POR_CICLO}", P / LECTURAS_POR_CICLO, "s"),
    ("Lecturas del logger en la campaña", int(D_s // intervalo_logger_s), "lecturas"),
    ("Frecuencia que verá el registro", round(frec_aparente(3600 / P, 3600 / intervalo_logger_s), 3), "ciclos/h"),
    ("Veredicto", veredicto(intervalo_logger_s, P), ""),
]
tabla = pd.DataFrame(filas, columns=["Concepto", "Colab", "Unidad"])
display(tabla)
print("\nPara pegar en Sheets (selecciona las 5 líneas, copia y pega en la celda E de la primera fila):\n")
for _, v, _ in filas:
    print(v)


## Siguiente paso: la campaña
1. Coloca el logger donde vive tu variable y llena la **ficha de campaña**.
2. Toma una foto del logger colocado.
3. No lo muevas durante 7 días. El próximo viernes exportas el CSV desde la app y lo subes a Classroom.